# 06 汇总判定

前五个 Notebook 已把每条声明写入 `reports/claims/<ID>.json`。本 Notebook：

1. 运行 Tier C 的登记项（它们会立即返回 `NOT_REPRODUCIBLE` 并附带原因）；
2. 汇总全部 39 条声明，统计 verdict；
3. 生成 `verdicts.json`、`report.md`、`report.html`。

> 判定不是“论文好/坏”的评分。它回答的是：**这句话在给定资源下能否被独立复核，复核结果是否与原文一致。**


In [1]:
import json
import os
import sys
from pathlib import Path

# 定位仓库根目录：容器内是 /app，本地打开时可能是 notebooks/ 或其子目录。
ROOT = Path(os.environ.get("EBBA_ROOT", Path.cwd())).resolve()
if not (ROOT / "src" / "ebba").exists():
    for cand in [Path.cwd(), *Path.cwd().parents]:
        if (cand / "src" / "ebba").exists():
            ROOT = cand
            break
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("EBBA_ROOT", str(ROOT))

import matplotlib.pyplot as plt
import numpy as np

from ebba import config, nbutil
from ebba.claims import load_registry, run, run_many

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("project root:", ROOT)
print("artifacts   :", config.ARTIFACTS_DIR)
print("reports     :", config.REPORTS_DIR)
print("fast mode   :", config.FAST)


project root: /app
artifacts   : /app/artifacts
reports     : /app/reports
fast mode   : False


## Tier C：容器内无法复现的条目

这些条目没有实现上的失败，只是需要论文级算力或未公开的数据/代码。显式记录它们，
是为了让报告不出现 PENDING，也避免把“没做”误读为“通过”。


In [2]:
c_results = run_many(["C01", "C02", "C03", "C04", "C05"])
for cid, res in c_results.items():
    print(f"{cid}: {res.verdict.value} — {res.notes_zh[:80]}...")


 C01  NOT_REPRODUCIBLE  Table 3/4 的全量数字（ResNet18、200 epochs、三数据集）
 C02  NOT_REPRODUCIBLE  语音 ESC-50 与文本 THUCNews 的后门检测结果
 C03  NOT_REPRODUCIBLE  13 组自适应攻击参数的完整训练复现
 C04  NOT_REPRODUCIBLE  all-to-all 与 full-target 攻击的完整评估
 C05  NOT_REPRODUCIBLE  ImageNet 子集（100 类、224×224）规模的结果
C01: NOT_REPRODUCIBLE — Table 3/4 的全量数字需要 ResNet18、200 epochs、CIFAR-10/GTSRB/ImageNet 三个数据集与五种攻击（BadNets...
C02: NOT_REPRODUCIBLE — ESC-50 语音与 THUCNews 文本实验需要对应数据集、语音/文本骨干网络与任务相关的预处理流程；这些资源不在容器内，且与图像管线的接口不同。...
C03: NOT_REPRODUCIBLE — 13 组自适应软标签攻击需要为每组参数重新训练被攻击模型，且攻击实现未公开；容器只能对论文已报告的 13 个数值做算术审计（A19）。...
C04: NOT_REPRODUCIBLE — all-to-all 与 full-target 攻击需要修改投毒标签生成器并对每种攻击重新训练；论文只给出结论性描述，缺少足够细节逐项复算。...
C05: NOT_REPRODUCIBLE — ImageNet 子集（100 类、224×224）需要 ImageNet 训练数据与 ResNet 级模型，下载量与算力都超出轻量容器的设计目标。...


## 全部声明一览


In [3]:
import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt

from ebba.claims.model import load_registry

registry = load_registry()
rows = []
for cid in registry:
    path = config.REPORTS_DIR / "claims" / f"{cid}.json"
    if not path.exists():
        rows.append({"id": cid, "tier": registry[cid]["tier"], "verdict": "PENDING",
                     "title_zh": registry[cid]["title_zh"]})
        continue
    payload = json.loads(path.read_text(encoding="utf-8"))
    rows.append({"id": cid, "tier": payload["tier"], "verdict": payload["verdict"],
                 "title_zh": payload["title_zh"]})

nbutil.show_table([r["id"] for r in rows])
print("rows:", len(rows), "| PENDING:", sum(r["verdict"] == "PENDING" for r in rows))


ID,Tier,Verdict,声明
A01,A,CONTRADICTED · 被否定,附录 B 算例：softmax 与能量数值是否可复算
A02,A,CONFIRMED · 已确认,Eq.3 是“最大值置 0”，而非减去最大值
A03,A,CONTRADICTED · 被否定,Eq.5、官方 README 与官方代码的归一化公式一致
A04,A,CONTRADICTED · 被否定,Eq.7 的求和下标与方差分母自洽
A05,A,CONFIRMED · 已确认,Eq.9-10 的 TE/NTE 恒等式及均值为 0
A06,A,CONTRADICTED · 被否定,Eq.9-10 与 Eq.4 使用同一温度 T
A07,A,PARTIAL · 部分成立 / 代理支持,Lemma 2 证明中 Eq.30 的代数等价性
A08,A,CONTRADICTED · 被否定,Lemma 2 对每一种损坏都成立
A09,A,CONTRADICTED · 被否定,Lemma 1（脏标签）的 NTK 核和占优
A10,A,CONTRADICTED · 被否定,Lemma 1（干净标签）的 NTK 核和占优


rows: 39 | PENDING: 0


## 判定统计与图


In [4]:
counts = Counter(r["verdict"] for r in rows)
order = ["CONFIRMED", "PARTIAL", "CONTRADICTED", "NOT_REPRODUCIBLE", "ERROR", "PENDING"]
for key in order:
    print(f"{key:<18} {counts.get(key, 0)}")

colors = {"CONFIRMED": "#1b7f3a", "PARTIAL": "#9a6700", "CONTRADICTED": "#b42318",
          "NOT_REPRODUCIBLE": "#5f6b7a", "ERROR": "#7a1fa2", "PENDING": "#c9d1d9"}
fig, ax = plt.subplots(figsize=(6.5, 3.2))
keys = [k for k in order if counts.get(k, 0)]
ax.bar(keys, [counts[k] for k in keys], color=[colors[k] for k in keys])
ax.set_ylabel("claims")
ax.set_title("Verdict distribution")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()


CONFIRMED          3
PARTIAL            13
CONTRADICTED       18
NOT_REPRODUCIBLE   5
ERROR              0
PENDING            0


## Tier 分布


In [5]:
by_tier = {}
for r in rows:
    by_tier.setdefault(r["tier"], Counter())[r["verdict"]] += 1
for tier in sorted(by_tier):
    print(tier, dict(by_tier[tier]))


A {'CONTRADICTED': 15, 'CONFIRMED': 3, 'PARTIAL': 2}
B {'PARTIAL': 11, 'CONTRADICTED': 3}
C {'NOT_REPRODUCIBLE': 5}


## 被否定的声明

下面逐条列出 `CONTRADICTED` 的实测要点。完整实测值在 `reports/report.html` 的折叠块中。


In [6]:
for r in rows:
    if r["verdict"] != "CONTRADICTED":
        continue
    payload = json.loads((config.REPORTS_DIR / "claims" / f'{r["id"]}.json').read_text(encoding="utf-8"))
    print(f'{r["id"]}  {r["title_zh"]}\n   {payload["notes_zh"]}\n')


A01  附录 B 算例：softmax 与能量数值是否可复算
   按论文自己给出的 logits 精确计算 softmax，s2 应为 [0.0067, 0.9933, 0.0000]，而非 [0.12, 0.88, 0]；能量为 [4.7243, 4.7185, 4.6691]，而非 [4.82, 4.43, 4.62]。原文的顺序 E0>E2>E1 与“目标标签能量最低”的结论在其自身 logits 下均不成立（精确顺序为 E0>E1>E2）。clamp 后的示例方向正确，但 s* 与 E* 仍有 0.01-0.04 的误差。

A03  Eq.5、官方 README 与官方代码的归一化公式一致
   论文 Eq.5、官方 README（CIFAR-10 用 P_k−1）与官方 Notebook（截断 2 再减 0.6）是三种不同归一化，不能同时成立。结论稳健性单独报告：三者选出的目标标签是否一致。

A04  Eq.7 的求和下标与方差分母自洽
   Eq.2/6 写作 k∈[0,K]（K+1 项），Eq.7 却除以 K。K 个真实标签时应使用 K 项、除数 K；若按字面补一项，最大 z-score 会明显变化。实现采用 K 项。

A06  Eq.9-10 与 Eq.4 使用同一温度 T
   Eq.4 的能量是 exp(s·T)/T，而 Eq.9/10 的比值把 T 消掉了（等价于 T=1）。形式不一致；实际影响用 argmax 一致率衡量。

A08  Lemma 2 对每一种损坏都成立
   Lemma 2 的原文是逐图逐损坏的普遍断言（st>s'_t 且 s_k1<s'_k1）。在缩比后门模型上统计全部 80 种损坏：同时满足两条的 (图,损坏) 比例为 76.01%，故普遍形式被反例否定；但按损坏类型取平均后，70.0% 的损坏呈现聚合方向，与论文插图想表达的总体趋势一致。

A09  Lemma 1（脏标签）的 NTK 核和占优
   按 Eq.13 的 NTK 核 K=e^{-2γ||x-x'||²}，在真实 CIFAR-10 像素上计算目标类与其他类的核和；γ 由查询-训练距离中位数标定，并报告 0.5×/1×/2× 的敏感性。两个情形分开看：目标类样本数多于其他类（190 vs 100）时目标核和几乎总是最大；两类样本数相等（100 v

## 部分成立与不可复现

`PARTIAL` 有两种来源：证明前提不完整（A07、A19），以及缩比实证只支持方向（所有 B 声明）。
`NOT_REPRODUCIBLE` 是 Tier C，附有所需资源说明。


In [7]:
for r in rows:
    if r["verdict"] in {"PARTIAL", "NOT_REPRODUCIBLE"}:
        print(f'{r["id"]:>4}  {r["verdict"]:<18} {r["title_zh"]}')


 A07  PARTIAL            Lemma 2 证明中 Eq.30 的代数等价性
 A19  PARTIAL            自适应攻击下 EBBA 的检测率超过 99%
 B01  PARTIAL            目标标签在干净/OOD 样本的 top 输出中更常见（Fig.1）
 B02  PARTIAL            EBBA 无需任务样本即可检测后门模型（Table 2）
 B03  PARTIAL            λ 的可用区间形状与 Fig.80 一致
 B04  PARTIAL            图像损坏使真值标签前移、目标标签后移（Fig.2）
 B05  PARTIAL            NTE 的 argmax 能恢复中毒图像的原始标签（Eq.12）
 B07  PARTIAL            EBBA+ 的 PDR/F1 达到论文报告的量级（Table 3）
 B09  PARTIAL            EBBA+ 去除后门且保持良性精度（Table 4）
 B10  PARTIAL            缩比模型上 BA ≥ (K-1)/K × ba
 B11  PARTIAL            均衡选择任务无关图像可减少误检（Fig.78/79）
 B12  PARTIAL            自适应软标签攻击下，双边检验 |E-μ|>λσ 仍可检测
 B13  PARTIAL            多目标攻击下能量统计可同时标出两个目标
 C01  NOT_REPRODUCIBLE   Table 3/4 的全量数字（ResNet18、200 epochs、三数据集）
 C02  NOT_REPRODUCIBLE   语音 ESC-50 与文本 THUCNews 的后门检测结果
 C03  NOT_REPRODUCIBLE   13 组自适应攻击参数的完整训练复现
 C04  NOT_REPRODUCIBLE   all-to-all 与 full-target 攻击的完整评估
 C05  NOT_REPRODUCIBLE   ImageNet 子集（100 类、224×224）规模的结果


## 运行耗时与环境

`reports/run_metadata.json` 记录每个 Notebook 的状态与耗时；`reports/verdicts.json` 是机器可读的最终产物。


In [8]:
meta_path = config.REPORTS_DIR / "run_metadata.json"
if meta_path.exists():
    meta = json.loads(meta_path.read_text(encoding="utf-8"))
    print("device:", meta.get("device"), "| fast:", meta.get("fast"),
          "| total:", meta.get("total_seconds"), "s")
    for rec in meta["notebooks"]:
        print(f'  {rec["name"]:<18} {rec["status"]:<8} {rec.get("seconds", "-")}s')
else:
    print("run_metadata.json 不存在（直接打开 Notebook 时正常）")


device: cpu | fast: False | total: 142.0 s
  00_overview        ok       2.8s
  01_formula_audit   ok       1.5s
  02_theory          ok       19.7s
  03_table_audit     ok       1.4s
  04_detection       ok       16.3s
  05_removal         ok       98.4s
  06_verdicts        ok       1.8s


## 结论怎么引用

1. **Tier A 的 `CONTRADICTED`** 是可直接复核的事实：附录 B 数值、表格算术、BA 下界反例、
   ASR 门槛反例、GTSRB 笔误、Table 3/4 过度声明、官方代码与论文协议不符。引用论文时应注明这些例外。
2. **Tier B 的 `PARTIAL`** 只说明方法机制在缩比设定下按预期工作，**不能**写成“复现了论文的 95% PDR”。
3. **Tier C 的 `NOT_REPRODUCIBLE`** 是范围声明，不是负面结论。
4. 任何 `ERROR` 或 `PENDING` 都应视为交付缺陷；`verify` 的退出码会反映执行故障。

最终报告：`reports/report.html`（自包含，可离线打开）与 `reports/verdicts.json`。
